In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Loading dataset...")
df = pd.read_csv("/content/drive/MyDrive/Spotify_MASTER_Dataset.csv")

# Define the 8 target acoustic features (excluding tempo)
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence'
]

# Remove rows with missing values in features or popularity
df = df.dropna(subset=features + ['popularity'])

# ==========================================
# Critical Addition: Oversampling for the Surrogate Model
# ==========================================
high_pop = df[df['popularity'] >= 75]
low_pop = df[df['popularity'] <= 25]

# Duplicate extreme edge cases (hits and flops) 3 times.
# This ensures the surrogate model learns the characteristics of extreme values
# comprehensively and prevents it from simply predicting the median for all songs.
df = pd.concat([df, high_pop, high_pop, high_pop, low_pop, low_pop, low_pop]).reset_index(drop=True)
print("Dataset balanced! Added duplicated extreme cases.")
# ==========================================

# Normalize the feature vectors using Min-Max scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])

# Normalize the target popularity scores to a [0, 1] range
y_scaled = df['popularity'].values / 100.0

class TabularSpotifyDataset(Dataset):
    """
    A PyTorch Dataset class designed for tabular Spotify data.
    It serves as the data loader for the Surrogate Model, matching
    normalized acoustic features to their corresponding popularity scores.
    """
    def __init__(self, X, y):
        """
        Initializes the TabularSpotifyDataset.

        Args:
            X (np.ndarray): The normalized feature matrix.
            y (np.ndarray): The normalized target popularity scores.
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1) # Shape: [batch_size, 1]

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        Returns:
            int: Number of data samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Retrieves a single feature-target pair from the dataset.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing:
                - X (torch.Tensor): The 8-dimensional feature vector.
                - y (torch.Tensor): The target popularity score (1-dimensional).
        """
        return self.X[idx], self.y[idx]

# Create the dataset instance
dataset = TabularSpotifyDataset(X_scaled, y_scaled)

# Split the dataset into 80% training and 20% validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Initialize PyTorch DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"✅ Data Ready! Features count: {len(features)}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")

Mounted at /content/drive
Loading dataset...
Dataset balanced! Added duplicated extreme cases.
✅ Data Ready! Features count: 8
Training samples: 20264, Validation samples: 5066


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Loading dataset...")
df = pd.read_csv("/content/drive/MyDrive/Spotify_MASTER_Dataset.csv")

# Define the 8 target acoustic features
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence'
]

# Remove rows with missing values in features or popularity
df = df.dropna(subset=features + ['popularity'])

# ==========================================
# Critical Addition: Oversampling for the Surrogate Model
# ==========================================
high_pop = df[df['popularity'] >= 75]
low_pop = df[df['popularity'] <= 25]

# Duplicate extreme edge cases (hits and flops) 3 times.
# This ensures the surrogate model learns the characteristics of extreme values
# comprehensively and prevents it from simply predicting the median for all songs.
df = pd.concat([df, high_pop, high_pop, high_pop, low_pop, low_pop, low_pop]).reset_index(drop=True)
print("Dataset balanced! Added duplicated extreme cases.")
# ==========================================

# Normalize the feature vectors using Min-Max scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])

# Normalize the target popularity scores to a [0, 1] range
y_scaled = df['popularity'].values / 100.0

class TabularSpotifyDataset(Dataset):
    """
    A PyTorch Dataset class designed for tabular Spotify data.
    It serves as the data loader for the Surrogate Model, matching
    normalized acoustic features to their corresponding popularity scores.
    """
    def __init__(self, X, y):
        """
        Initializes the TabularSpotifyDataset.

        Args:
            X (np.ndarray): The normalized feature matrix.
            y (np.ndarray): The normalized target popularity scores.
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1) # Shape: [batch_size, 1]

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        Returns:
            int: Number of data samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Retrieves a single feature-target pair from the dataset.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing:
                - X (torch.Tensor): The 8-dimensional feature vector.
                - y (torch.Tensor): The target popularity score (1-dimensional).
        """
        return self.X[idx], self.y[idx]

# Create the dataset instance
dataset = TabularSpotifyDataset(X_scaled, y_scaled)

# Split the dataset into 80% training and 20% validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Initialize PyTorch DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"✅ Data Ready! Features count: {len(features)}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")


# ==========================================
# Surrogate Model Definition & Training Loop
# ==========================================

class SurrogateMLP(nn.Module):
    """
    A Multi-Layer Perceptron (MLP) acting as a surrogate environment model.
    It maps a set of numerical audio features directly to a predicted popularity score.
    This allows the reinforcement learning agent to simulate and evaluate
    mixing adjustments rapidly without relying on the heavier, full multimodal network.
    """
    def __init__(self, input_dim):
        """
        Initializes the SurrogateMLP architecture.

        Args:
            input_dim (int): The number of input audio features.
        """
        super(SurrogateMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid() # Normalize the output score to a [0, 1] range
        )

    def forward(self, x):
        """
        Forward pass of the surrogate model.

        Args:
            x (torch.Tensor): A batch of input acoustic feature vectors.

        Returns:
            torch.Tensor: A batch of predicted popularity scores.
        """
        return self.net(x)

# Set up hardware device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the surrogate model
surrogate_model = SurrogateMLP(input_dim=len(features)).to(device)

# Define training hyperparameters and optimization criteria
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(surrogate_model.parameters(), lr=0.001)
epochs = 20

print("🚀 Starting Surrogate Training...")

best_val_loss = float('inf')

# Main training loop
for epoch in range(epochs):
    # --- Training Phase ---
    surrogate_model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = surrogate_model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # --- Validation Phase ---
    surrogate_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = surrogate_model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Model Checkpointing ---
    # Save the model state if the validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(surrogate_model.state_dict(), "/content/drive/MyDrive/Spotify_Project_Data/best_surrogate_model.pth")

print("🎉 Surrogate Training Complete! Model saved.")

🚀 Starting Surrogate Training...
Epoch 1/20 | Train Loss: 0.0181 | Val Loss: 0.0144
Epoch 2/20 | Train Loss: 0.0156 | Val Loss: 0.0139
Epoch 3/20 | Train Loss: 0.0153 | Val Loss: 0.0141
Epoch 4/20 | Train Loss: 0.0151 | Val Loss: 0.0138
Epoch 5/20 | Train Loss: 0.0150 | Val Loss: 0.0135
Epoch 6/20 | Train Loss: 0.0149 | Val Loss: 0.0136
Epoch 7/20 | Train Loss: 0.0148 | Val Loss: 0.0137
Epoch 8/20 | Train Loss: 0.0148 | Val Loss: 0.0135
Epoch 9/20 | Train Loss: 0.0147 | Val Loss: 0.0135
Epoch 10/20 | Train Loss: 0.0146 | Val Loss: 0.0135
Epoch 11/20 | Train Loss: 0.0145 | Val Loss: 0.0135
Epoch 12/20 | Train Loss: 0.0145 | Val Loss: 0.0135
Epoch 13/20 | Train Loss: 0.0144 | Val Loss: 0.0134
Epoch 14/20 | Train Loss: 0.0144 | Val Loss: 0.0134
Epoch 15/20 | Train Loss: 0.0144 | Val Loss: 0.0134
Epoch 16/20 | Train Loss: 0.0143 | Val Loss: 0.0133
Epoch 17/20 | Train Loss: 0.0142 | Val Loss: 0.0133
Epoch 18/20 | Train Loss: 0.0143 | Val Loss: 0.0133
Epoch 19/20 | Train Loss: 0.0141 | Val L

In [ ]:
!pip install stable-baselines3 gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MinMaxScaler
from stable_baselines3 import PPO
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Loading dataset...")
df = pd.read_csv("/content/drive/MyDrive/Spotify_MASTER_Dataset.csv")

# Define the 8 target acoustic features
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence'
]

# Remove rows with missing values in features or popularity
df = df.dropna(subset=features + ['popularity'])

# ==========================================
# Critical Addition: Oversampling for the Surrogate Model
# ==========================================
high_pop = df[df['popularity'] >= 75]
low_pop = df[df['popularity'] <= 25]

# Duplicate extreme edge cases (hits and flops) 3 times.
# This ensures the surrogate model learns the characteristics of extreme values
# comprehensively and prevents it from simply predicting the median for all songs.
df = pd.concat([df, high_pop, high_pop, high_pop, low_pop, low_pop, low_pop]).reset_index(drop=True)
print("Dataset balanced! Added duplicated extreme cases.")
# ==========================================

# Normalize the feature vectors using Min-Max scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])

# Normalize the target popularity scores to a [0, 1] range
y_scaled = df['popularity'].values / 100.0

class TabularSpotifyDataset(Dataset):
    """
    A PyTorch Dataset class designed for tabular Spotify data.
    It serves as the data loader for the Surrogate Model, matching
    normalized acoustic features to their corresponding popularity scores.
    """
    def __init__(self, X, y):
        """
        Initializes the TabularSpotifyDataset.

        Args:
            X (np.ndarray): The normalized feature matrix.
            y (np.ndarray): The normalized target popularity scores.
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1) # Shape: [batch_size, 1]

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        Returns:
            int: Number of data samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Retrieves a single feature-target pair from the dataset.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing:
                - X (torch.Tensor): The 8-dimensional feature vector.
                - y (torch.Tensor): The target popularity score (1-dimensional).
        """
        return self.X[idx], self.y[idx]

# Create the dataset instance
dataset = TabularSpotifyDataset(X_scaled, y_scaled)

# Split the dataset into 80% training and 20% validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Initialize PyTorch DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"✅ Data Ready! Features count: {len(features)}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")


# ==========================================
# Surrogate Model Definition & Training Loop
# ==========================================

class SurrogateMLP(nn.Module):
    """
    A Multi-Layer Perceptron (MLP) acting as a surrogate environment model.
    It maps a set of numerical audio features directly to a predicted popularity score.
    This allows the reinforcement learning agent to simulate and evaluate
    mixing adjustments rapidly without relying on the heavier, full multimodal network.
    """
    def __init__(self, input_dim):
        """
        Initializes the SurrogateMLP architecture.

        Args:
            input_dim (int): The number of input audio features.
        """
        super(SurrogateMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid() # Normalize the output score to a [0, 1] range
        )

    def forward(self, x):
        """
        Forward pass of the surrogate model.

        Args:
            x (torch.Tensor): A batch of input acoustic feature vectors.

        Returns:
            torch.Tensor: A batch of predicted popularity scores.
        """
        return self.net(x)

# Set up hardware device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the surrogate model
surrogate_model = SurrogateMLP(input_dim=len(features)).to(device)

# Define training hyperparameters and optimization criteria
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(surrogate_model.parameters(), lr=0.001)
epochs = 20

print("🚀 Starting Surrogate Training...")

best_val_loss = float('inf')

# Main training loop
for epoch in range(epochs):
    # --- Training Phase ---
    surrogate_model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = surrogate_model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # --- Validation Phase ---
    surrogate_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = surrogate_model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Model Checkpointing ---
    # Save the model state if the validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(surrogate_model.state_dict(), "/content/drive/MyDrive/Spotify_Project_Data/best_surrogate_model.pth")

print("🎉 Surrogate Training Complete! Model saved.")


# ==========================================
# Reinforcement Learning: Virtual Producer Environment
# ==========================================

class RealisticSongOptimizerEnv(gym.Env):
    """
    A custom Gymnasium environment simulating a professional music studio.
    The environment allows an RL agent to adjust 8 acoustic features to maximize
    the song's predicted popularity score, while heavily penalizing unrealistic changes.
    """
    def __init__(self, model, X_data):
        """
        Initializes the RL environment.

        Args:
            model (torch.nn.Module): The trained surrogate model to predict scores.
            X_data (np.ndarray): The dataset of normalized song features to sample from.
        """
        super(RealisticSongOptimizerEnv, self).__init__()
        self.model = model
        self.X_data = X_data

        # Action Space: 17 discrete actions
        # Actions 0-7: Increase feature by 0.05
        # Actions 8-15: Decrease feature by 0.05
        # Action 16: Do nothing (Stop)
        self.action_space = spaces.Discrete(17)

        # Observation Space: 8 normalized acoustic features (0.0 to 1.0)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(8,), dtype=np.float32)

        self.current_state = None
        self.initial_state = None
        self.current_score = 0
        self.steps = 0
        self.max_steps = 15

    def _get_score(self, state):
        """
        Evaluates the current feature state using the surrogate model.

        Args:
            state (np.ndarray): The current 8-feature state vector.

        Returns:
            float: The predicted popularity score (0 to 100).
        """
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            return self.model(state_tensor).item() * 100

    def reset(self, seed=None):
        """
        Resets the environment to a random song from the dataset for a new episode.
        """
        idx = np.random.randint(len(self.X_data))
        self.current_state = np.copy(self.X_data[idx])
        self.initial_state = np.copy(self.current_state)
        self.current_score = self._get_score(self.current_state)
        self.steps = 0
        return self.current_state, {}

    def step(self, action):
        """
        Executes a single studio adjustment based on the agent's action.

        Args:
            action (int): The chosen discrete action index.

        Returns:
            tuple: (new_state, reward, terminated, truncated, info_dict)
        """
        self.steps += 1
        new_state = np.copy(self.current_state)
        reward = 0
        penalty = 0

        # --- Studio Rules & Constraints ---
        if action < 8: # Increase action (indices 0 to 7)
            if new_state[action] >= 0.95:
                penalty += 1.0 # Penalize attempting to push beyond maximum logical bounds
            else:
                new_state[action] += 0.05

        elif action < 16: # Decrease action (indices 8 to 15)
            feature_idx = action - 8
            if new_state[feature_idx] <= 0.05:
                penalty += 1.0 # Penalize attempting to push below minimum logical bounds
            else:
                new_state[feature_idx] -= 0.05

        # Action 16 is "Stop/Do Nothing", which causes no state change and no penalty.

        new_state = np.clip(new_state, 0.0, 1.0)

        new_score = self._get_score(new_state)
        base_reward = new_score - self.current_score

        # Distance Penalty: Encourage creativity but penalize completely ruining the original song's identity
        distance_from_original = np.linalg.norm(new_state - self.initial_state)
        distance_penalty = distance_from_original * 1.5

        reward = base_reward - penalty - distance_penalty

        terminated = False
        truncated = self.steps >= self.max_steps

        self.current_state = new_state
        self.current_score = new_score

        return self.current_state, reward, terminated, truncated, {}

# --- Agent Training ---
print("🎮 Creating Realistic Gym Environment (8 Features)...")
realistic_env = RealisticSongOptimizerEnv(model=surrogate_model, X_data=X_scaled)

print("🤖 Initializing Producer PPO Agent...")
smart_agent = PPO("MlpPolicy", realistic_env, verbose=1, learning_rate=0.0003)

print("🚀 Training the Agent (Playing the game 60,000 times)...")
smart_agent.learn(total_timesteps=60000)

print("✅ Training Complete! Saving Agent...")
agent_path_v5 = "/content/drive/MyDrive/Spotify_Project_Data/hit_optimizer_agent_v5"
smart_agent.save(agent_path_v5)
print(f"Smart Agent saved as '{agent_path_v5}.zip'")

🎮 Creating Realistic Gym Environment (8 Features)...
🤖 Initializing Producer PPO Agent...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🚀 Training the Agent (Playing the game 60,000 times)...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 15       |
|    ep_rew_mean     | -4.19    |
| time/              |          |
|    fps             | 1124     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 15          |
|    ep_rew_mean          | -1.24       |
| time/                   |             |
|    fps                  | 771         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
